# 3.3 Face Trial 003: MobileNetV2 + Balanced Sampling

Purpose:
- Establish a controlled lightweight transfer-learning baseline.
- Use class-balanced tf.data sampling to prevent majority-class collapse.
- Evaluate on the untouched test set.

Trial 001 and Trial 002 showed that AutoKeras pipelines run technically, but the selected models collapsed to majority-class prediction. Trial 003 tests whether a controlled pretrained lightweight model can recover Stroke-class learning.

In [1]:
import gc
import math
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FACE_SPLIT_MANIFEST_PATH,
    FACE_IMAGE_SIZE,
    FACE_BATCH_SIZE,
    FACE_TRIAL_003_MODEL_PATH,
    FACE_TRIAL_003_RESULTS_PATH,
    FACE_TRIAL_003_PREDICTIONS_PATH,
    FACE_TRIAL_003_CONFUSION_MATRIX_PATH,
    FACE_TRIAL_003_HISTORY_PATH,
)

from rural_stroke_assist.preprocessing.tf_image_dataset import (
    create_balanced_binary_training_dataset,
    dataframe_to_tf_dataset,
)

from rural_stroke_assist.preprocessing.face_dataset import (
    INVERSE_LABEL_MAPPING,
)

from rural_stroke_assist.modeling.vision_models import (
    build_mobilenetv2_binary_classifier,
)

from rural_stroke_assist.modeling.callbacks import (
    build_early_stopping_callback,
    build_csv_logger_callback,
)

from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

In [2]:
tf.keras.backend.clear_session()
gc.collect()

tf.random.set_seed(42)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Enabled memory growth for {gpu}")
    except Exception as exc:
        print(f"Could not set memory growth for {gpu}: {exc}")


GPUs: []


In [3]:
face_split_df = pd.read_csv(FACE_SPLIT_MANIFEST_PATH)

face_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   NonStroke      208
       Stroke         110
train  NonStroke      970
       Stroke         510
val    NonStroke      208
       Stroke         109
dtype: int64

In [4]:
train_df = face_split_df[face_split_df["split"] == "train"].copy()
val_df = face_split_df[face_split_df["split"] == "val"].copy()
test_df = face_split_df[face_split_df["split"] == "test"].copy()

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 1480
Val: 317
Test: 318


In [5]:
IMAGE_SIZE = FACE_IMAGE_SIZE  # currently (160, 160)
BATCH_SIZE = 16
EPOCHS = 20
PATIENCE = 4

steps_per_epoch = math.ceil(len(train_df) / BATCH_SIZE)

IMAGE_SIZE, BATCH_SIZE, EPOCHS, PATIENCE, steps_per_epoch

((160, 160), 16, 20, 4, 93)

In [6]:
train_ds = create_balanced_binary_training_dataset(
    df=train_df,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=42,
)

val_ds = dataframe_to_tf_dataset(
    val_df,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = dataframe_to_tf_dataset(
    test_df,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [7]:
for images, labels in train_ds.take(3):
    unique, counts = np.unique(labels.numpy(), return_counts=True)
    print(dict(zip(unique, counts)))
    print(images.shape, float(tf.reduce_min(images)), float(tf.reduce_max(images)))

{0: 8, 1: 8}
(16, 160, 160, 3) 0.0 1.0
{0: 7, 1: 9}
(16, 160, 160, 3) 0.0 1.0
{0: 7, 1: 9}
(16, 160, 160, 3) 0.0 1.0


In [8]:
model = build_mobilenetv2_binary_classifier(
    image_size=IMAGE_SIZE,
    dropout_rate=0.3,
    learning_rate=1e-4,
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "mobilenetv2_face_binary_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
callbacks = [
    build_early_stopping_callback(
        patience=PATIENCE,
        monitor="val_loss",
    ),
    build_csv_logger_callback(
        output_path=FACE_TRIAL_003_HISTORY_PATH,
    ),
]

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks,
)

Epoch 1/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 19s 158ms/step - accuracy: 0.5612 - auc: 0.5838 - loss: 0.7699 - precision: 0.5895 - recall: 0.5078 - val_accuracy: 0.5741 - val_auc: 0.6363 - val_loss: 0.7309 - val_precision: 0.4278 - val_recall: 0.7064
Epoch 2/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - accuracy: 0.6163 - auc: 0.6569 - loss: 0.6883 - precision: 0.6284 - recall: 0.6110 - val_accuracy: 0.6215 - val_auc: 0.7112 - val_loss: 0.6876 - val_precision: 0.4706 - val_recall: 0.8073
Epoch 3/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - accuracy: 0.6606 - auc: 0.7075 - loss: 0.6305 - precision: 0.6618 - recall: 0.7073 - val_accuracy: 0.6688 - val_auc: 0.7671 - val_loss: 0.6332 - val_precision: 0.5112 - val_recall: 0.8349
Epoch 4/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 11s 123ms/step - accuracy: 0.6741 - auc: 0.7400 - loss: 0.6004 - precision: 0.6582 - recall: 0.7313 - val_accuracy: 0.7287 - val_auc: 0.8087 - val_loss: 0.5555 - val_precision: 0.5804 - val_recall: 0.7615
Epoch 5/20
93/93 ━━━━━━━━━━━

In [11]:
raw_scores = model.predict(test_ds)

y_score = raw_scores.reshape(-1)
y_pred = (y_score >= 0.5).astype(int)

y_test = test_df["class_label"].map({"NonStroke": 0, "Stroke": 1}).to_numpy()

print("Pred counts:", np.unique(y_pred, return_counts=True))
print("True counts:", np.unique(y_test, return_counts=True))

20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 241ms/step
Pred counts: (array([0, 1]), array([207, 111], dtype=int64))
True counts: (array([0, 1], dtype=int64), array([208, 110], dtype=int64))


In [12]:
metrics = evaluate_binary_classifier(
    y_true=y_test,
    y_pred=y_pred,
    y_score=y_score,
)

metrics

{'accuracy': 0.9276729559748428,
 'precision': 0.8918918918918919,
 'recall': 0.9,
 'f1': 0.8959276018099548,
 'roc_auc': 0.9809003496503497}

In [13]:
target_names = [INVERSE_LABEL_MAPPING[0], INVERSE_LABEL_MAPPING[1]]

report = create_classification_report_dict(
    y_true=y_test,
    y_pred=y_pred,
    target_names=target_names,
)

report

{'NonStroke': {'precision': 0.9468599033816425,
  'recall': 0.9423076923076923,
  'f1-score': 0.944578313253012,
  'support': 208.0},
 'Stroke': {'precision': 0.8918918918918919,
  'recall': 0.9,
  'f1-score': 0.8959276018099548,
  'support': 110.0},
 'accuracy': 0.9276729559748428,
 'macro avg': {'precision': 0.9193758976367672,
  'recall': 0.9211538461538462,
  'f1-score': 0.9202529575314834,
  'support': 318.0},
 'weighted avg': {'precision': 0.9278458113568859,
  'recall': 0.9276729559748428,
  'f1-score': 0.9277494508041558,
  'support': 318.0}}

In [14]:
save_confusion_matrix_plot(
    y_true=y_test,
    y_pred=y_pred,
    class_names=target_names,
    output_path=FACE_TRIAL_003_CONFUSION_MATRIX_PATH,
)

FACE_TRIAL_003_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/face_trial_003_mobilenetv2_balanced_160_confusion_matrix.png')

In [15]:
FACE_TRIAL_003_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.copy()
predictions_df["true_label_encoded"] = y_test
predictions_df["stroke_probability"] = y_score
predictions_df["predicted_label_encoded"] = y_pred
predictions_df["true_label"] = predictions_df["true_label_encoded"].map(INVERSE_LABEL_MAPPING)
predictions_df["predicted_label"] = predictions_df["predicted_label_encoded"].map(INVERSE_LABEL_MAPPING)
predictions_df["is_correct"] = predictions_df["true_label_encoded"] == predictions_df["predicted_label_encoded"]

predictions_df.to_csv(FACE_TRIAL_003_PREDICTIONS_PATH, index=False)

FACE_TRIAL_003_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/face_trial_003_mobilenetv2_balanced_160_predictions.csv')

In [16]:
save_markdown_results(
    metrics=metrics,
    report=report,
    output_path=FACE_TRIAL_003_RESULTS_PATH,
    title="Face Trial 003 MobileNetV2 Balanced Sampling Results",
)

FACE_TRIAL_003_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/face_trial_003_mobilenetv2_balanced_160_results.md')

In [17]:
stroke_recall = report["Stroke"]["recall"]
stroke_f1 = report["Stroke"]["f1-score"]
roc_auc = metrics.get("roc_auc", 0)

accepted = (
    stroke_recall >= 0.50
    and stroke_f1 >= 0.40
    and roc_auc >= 0.65
)

{
    "stroke_recall": stroke_recall,
    "stroke_f1": stroke_f1,
    "roc_auc": roc_auc,
    "accepted": accepted,
}

{'stroke_recall': 0.9,
 'stroke_f1': 0.8959276018099548,
 'roc_auc': 0.9809003496503497,
 'accepted': True}

In [18]:
if accepted:
    FACE_TRIAL_003_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save(FACE_TRIAL_003_MODEL_PATH)
    print("Accepted model saved to:", FACE_TRIAL_003_MODEL_PATH)
else:
    print("Model rejected. Results and predictions saved, model artifact not saved.")

Accepted model saved to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\models\experiments\face\trial_003_mobilenetv2_balanced_160\model.keras
